In [2]:
!pip install transformers datasets scikit-learn -q

In [3]:
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


In [13]:
dataset = load_dataset("ag_news")

df_train = pd.DataFrame(dataset["train"])
df_test = pd.DataFrame(dataset["test"])

def clean_text(text):
  import re
  text = text.lower()
  text = re.sub(r"http\S+", "", text)
  text = re.sub(r"[^a-z0-9\s]", "", text)
  text = re.sub(r"\s+", "", text).strip()
  return text
df_train["clean_text"] = df_train["text"]
df_test["clean_text"] = df_test["text"]

df_train = df_train[df_train["clean_text"].str.strip().str.len() > 10].reset_index(drop=True)
df_test  = df_test[df_test["clean_text"].str.strip().str.len() > 10].reset_index(drop=True)

print(f"Train: {len(df_train)} | Test: {len(df_test)}")
print(df_train["clean_text"].iloc[0])

Train: 120000 | Test: 7600
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [14]:
train_dataset = Dataset.from_pandas(df_train[["clean_text", "label"]].reset_index(drop=True))
test_dataset  = Dataset.from_pandas(df_test[["clean_text", "label"]].reset_index(drop=True))

MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset  = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization done!")
print(train_dataset[0]["input_ids"][:20])

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

✅ Tokenization done!
tensor([  101,  2813,  2358,  1012,  6468, 15020,  2067,  2046,  1996,  2304,
         1006, 26665,  1007, 26665,  1011,  2460,  1011, 19041,  1010,  2813])


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)
print(f"✅ Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded! Parameters: 66,956,548


In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

In [17]:
training_args = TrainingArguments(
    output_dir="distilbert-agnews",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=100,
    fp16=True,
    report_to="none"
)
print("✅ Training arguments set!")

✅ Training arguments set!


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Starting training on FULL dataset with GPU...")
trainer.train()

Starting training on FULL dataset with GPU...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.200146,0.174671,0.942237,0.942311
2,0.114213,0.173588,0.946842,0.946871
3,0.063832,0.203768,0.946711,0.946700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=11250, training_loss=0.13713235886891684, metrics={'train_runtime': 1204.238, 'train_samples_per_second': 298.944, 'train_steps_per_second': 9.342, 'total_flos': 1.192249110528e+16, 'train_loss': 0.13713235886891684, 'epoch': 3.0})

In [19]:
results = trainer.evaluate()

print("\n📊 Final Results:")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Macro: {results['eval_f1_macro']:.4f}")


📊 Final Results:
Accuracy: 0.9470
F1 Macro: 0.9470


In [20]:
trainer.save_model("distilbert-agnews-final")
tokenizer.save_pretrained("distilbert-agnews-final")

print("✅ Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


In [21]:
import shutil
shutil.make_archive("distilbert-agnews-final", "zip", "distilbert-agnews-final")

from google.colab import files
files.download("distilbert-agnews-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# Check 1 — what does a training example look like?
print("First training example:")
print(train_dataset[0])

# Check 2 — what are the unique label values?
import pandas as pd
print("\nUnique labels in train:", pd.Series(train_dataset["labels"]).unique())
print("Label counts:\n", pd.Series(train_dataset["labels"]).value_counts())

# Check 3 — what does the raw dataframe look like?
print("\nRaw df_train head:")
print(df_train[["clean_text", "label"]].head())

First training example:
{'labels': tensor(2), 'input_ids': tensor([101, 100, 102,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0]), 'attention_mask': tensor([1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

ValueError: Length of values (58958) does not match length of index (58959)